# tSCS EMG — 30 Hz burst trains, changed polarity + lidocaine (NTA, 24-07-2026)

## What a burst file is
Each **stimulation window** is one **train**: 30 Hz × 1000 ms = **30 pulses**, 33.4 ms apart.
Every pulse evokes its own response. One file = one recruitment sweep = one train per intensity
(10, 15, … 50 mA). So a *train* and an *intensity* are the same thing here.

## What is measured — fully automatic, nothing picked by hand
For each train, the first **`N_PULSES`** pulses are analysed. For pulse *k*:

    window_k = [ pulse_k onset + RESP_START_MS ,  pulse_(k+1) onset − GUARD_MS ]
    p2p_k    = max(EMG) − min(EMG) inside window_k          (mV)

`RESP_START_MS` exists because the stimulus artifact outlasts the trigger pulse — by a different
amount on every channel. The diagnostics cell prints the measured artifact width per channel.

## Motor-response criterion (`MIN_SNR`)
Automatic, per train (= per muscle × intensity). Two numbers are compared:

1. **signal** — the pulse-1 peak-to-peak, measured in `window_1` (8 → 32.4 ms after the pulse)
2. **noise** — the peak-to-peak of the *pre-stimulus baseline* of that same trace, measured over
   windows of the **same length** (24.4 ms) placed between −95 and −5 ms, averaged

A train **has a motor response if signal ≥ `MIN_SNR` × noise** (default 3). In words: *the first
pulse must evoke something at least three times bigger than what the same window shows when
nothing is stimulated.* Trains that fail are marked *below criterion* and excluded from every
number. Pulse 1 is used because it is the only pulse not affected by depression from a previous
pulse. Only pulse 1 is tested — pulses 2–N of a passing train are kept whatever their size.

## Artifact rejection (`MAX_EDGE_FRAC`)
Some channels (Deltoid, Biceps, Triceps) show no EMG wave between pulses, only the smooth
recovery of the stimulus artifact. Such a curve has no peak inside the window: its max and min
fall **on the window borders**. So a train is **rejected as artifact when more than
`MAX_EDGE_FRAC` (50 %) of its pulses have their max or min within 1 ms of a border.** It is then
treated like a non-responding train and labelled *ARTIFACT – rejected* in the figures.

## Design of this notebook
**Nothing is averaged across intensities.** Each intensity is compared on its own,
changed polarity, with the raw traces and the detected peaks visible in every figure.

## Files — changed-polarity lidocaine session (subject NTA, 24-07-2026, `changedpol.xlsx`)
Electrode 2, anode at the iliac crests. Every protocol was run in **both polarities** —
**cathodic** (polarity 2) and **anodic** (polarity 1) — **before** and **after** lidocaine
(applied 45 min, ~10:50 → 11:33). Folder `testSCS` (the copy under `P04tscsHealthy/` is the
same first recording, mis-named before the control software was corrected).

| | single pulse | burst | ARC-EX (Modulated) |
|---|---|---|---|
| **pre · cathodic** | `100913` (10:09) | `102443` (10:24; `102236` aborted) | `103530` (10:35; `103040` no response) |
| **pre · anodic** | `101941` (10:19) | `102721` (10:27) | `103849` (10:38) |
| **post · cathodic** | `113447` (11:34) | `113834` (11:38) | `114332` (11:43) |
| **post · anodic** | `113632` (11:36) | `114055` (11:40) | `114730` (11:47; `114630` ignore) |

Motor threshold from the log — burst: cathodic 25 mA pre / 30 post, anodic 30 / 30;
ARC-EX: cathodic 70 / 70, anodic 65 / 90; single pulse: ~40 pre (anodic), 30–40 post.
Participant: *"less pain with anodic"*, *"much less pain after lidocaine"*.

Two questions in one notebook, chosen with `USE`: **polarity** (cathodic vs anodic, same state)
and **lidocaine** (pre vs post, same polarity).

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from functions import set_style, load_run, detect_pulses, pretty, waterfall, waterfall_overlay
from functions.burst import (diagnostics, compare_at_intensity, summary_curves,
                             burst_p2p, save_burst_csv, motor_threshold,
                             plot_pulse_overlay, detection_report)
set_style()


## 1 · Config

In [ ]:
D = "tSCS_CHUV_data/24-07-2026/testSCS/"
CONDITIONS = [   # (label, file) - index them in USE; plotted in that order: gray, orange, blue, green
    ("pre cathodic", "Burst_autosave_20260724_102443_670ms.csv"),
    ("pre anodic", "Burst_autosave_20260724_102721_694ms.csv"),
    ("post cathodic", "Burst_autosave_20260724_113834_522ms.csv"),
    ("post anodic", "Burst_autosave_20260724_114055_891ms.csv"),
]
USE = [0, 1]           # <-- e.g. [0, 1] polarity before lidocaine · [0, 2] lidocaine, cathodic
                       #         [1, 3] lidocaine, anodic          · [0, 1, 2, 3] everything

LABELS = [CONDITIONS[i][0] for i in USE]
CSVS   = [D + CONDITIONS[i][1] for i in USE]

N_PULSES      = 10     # first N pulses of each train
RESP_START_MS = 8.0    # response window starts this long after EACH pulse onset (must clear the artifact)
                       # can also be a dict to override single muscles after the reliability check, e.g.
                       # RESP_START_MS = {"R_DELmed": 11.0}  -> 11 ms for that channel, 8 ms for the rest
                       # (channel names = the keys of `sig`; the default for the rest is 8.0)
GUARD_MS      = 1.0    # ...and stops this long before the next pulse
MIN_SNR       = 2.0    # motor-response criterion: pulse-1 p2p >= MIN_SNR x baseline p2p (None = keep all)
MAX_EDGE_FRAC = 0.5    # ARTIFACT rejection: a train is thrown out when more than this fraction of its
                       # pulses have their max/min sitting on the window border (= smooth artifact
                       # recovery, no EMG wave - Deltoid / Biceps / Triceps here). None = off
AMP_MT        = 30     # motor threshold (mA) from the log; highlighted in the figures

KW = dict(n_pulses=N_PULSES, resp_start_ms=RESP_START_MS, guard_ms=GUARD_MS, min_snr=MIN_SNR,
          max_edge_frac=MAX_EDGE_FRAC)

RUNS    = [load_run(f) for f in CSVS]                       # [(meta, t, sig), ...]
muscles = [c for c in RUNS[0][2] if c != "Trigger A" and all(c in r[2] for r in RUNS)]
AMPS    = sorted(set.intersection(*[{m["amp_ma"] for m in r[0]} for r in RUNS]))   # in ALL selected files
for lab, f, (meta, t, sig) in zip(LABELS, CSVS, RUNS):
    print(f"{lab:18s} {f.split('/')[-1]} | electrode {meta[0]['electrode']} | {[m['amp_ma'] for m in meta]} mA")
AMP_SHOW = AMP_MT if AMP_MT in AMPS else max(AMPS)
print(f"compared at: {AMPS} mA | motor threshold: {AMP_MT} mA"
      + ("" if AMP_SHOW == AMP_MT else f"  (MT outside the common sweep -> diagnostics at {AMP_SHOW} mA)"))


## 2 · Diagnostics — is the detection sound?

Print-only, one block per condition. Read it top to bottom:

1. **artifact width per channel** — `RESP_START_MS` must be larger than these, otherwise the
   "response" is artifact decay. A `!!` line tells you which channel it does *not* clear.
2. **trains kept** per muscle — how many intensities pass the motor-response criterion and from
   which mA. This is the data-derived motor threshold *per muscle*.
3. **CLIPPED** — a channel whose amplifier saturated. Its peak-to-peak is meaningless.

Known for this session: the proximal channels (Deltoid, Biceps, Triceps) show a smooth
artifact-recovery curve between pulses rather than a discrete response — they pass the criterion
on artifact alone and should not be interpreted. **Thenar (R)** clips after lidocaine.

In [ ]:
RES = []
for lab, (meta, t, sig), f in zip(LABELS, RUNS, CSVS):
    print("=" * 28, lab, "=" * 28)
    RES.append(diagnostics(meta, t, sig, muscles, amp=AMP_SHOW, **KW))
    print("saved", save_burst_csv(RES[-1], muscles, f, meta=meta, normalize="none")); print()


## 3 · Reliability of the automatic peak-to-peak

Picking every peak by hand is not feasible (10 pulses × 15 muscles × 9 intensities × 2 files),
and an unchecked automatic pick is the risk. So the detection stays automatic and is **checked in
two ways**, then **corrected per muscle**, not per peak.

### 3a · Pulse overlay — the visual check
For one intensity, every pulse's segment is cut out and **re-aligned to its own pulse onset**
(t = 0), then all N are overlaid; colour = pulse number (dark = 1, light = N). Green = the response
window, ▼/▲ = the detected max/min of each pulse.

**If the detection is picking the same deflection every time, the markers stack on top of each
other.** A marker off on its own = a misdetection, and it gets a **red ring**. The panel title
turns red with the count.

### 3b · Automatic flags — the numeric check
Every detected max/min is tested on all responding trains:

- **edge** — it sits within 1 ms of a window border. The "peak" is the artifact tail, the next
  pulse's onset, or a wave the window cuts off. *A muscle where almost every pulse is "edge" is
  not giving you an EMG response — it is artifact recovery* (the proximal channels here).
- **jitter** — its time after its own pulse differs by > 3 ms from the train's median: it is not
  the same deflection as the other pulses (noise spike, movement, a different wave).

A muscle is trustworthy when its flagged count is low and the overlay stacks. Where a real
response is visible but the window cuts it, override `RESP_START_MS` for that channel in the
config (dict form) and re-run — that is the semi-manual step.

In [ ]:
for lab, (meta, t, sig) in zip(LABELS, RUNS):
    plot_pulse_overlay(meta, t, sig, muscles, amp=AMP_SHOW, title=f"{AMP_SHOW} mA - {lab}", **KW)


In [ ]:
for lab, res in zip(LABELS, RES):
    print("=" * 28, lab, "=" * 28)
    detection_report(res, muscles); print()


## 4 · Raw traces — waterfalls

One trace per intensity stacked at its amplitude, first **3 pulses** of the train (0, 33.4,
66.7 ms), red = artifact of each pulse. **Same gain per muscle in both figures** — the first call
returns the gains and the second reuses them, so a smaller response after lidocaine *draws*
smaller.

### 4a · First condition (sets the gain)

In [ ]:
XLIM_WF = (-20, 130)
gains = waterfall(*RUNS[0], muscles, xlim=XLIM_WF)          # first condition sets the gain


### 4b · The other conditions — same gain

In [ ]:
for lab, (meta, t, sig) in zip(LABELS[1:], RUNS[1:]):
    print(lab); waterfall(meta, t, sig, muscles, xlim=XLIM_WF, gains=gains)


### 4c · Overlay — all selected conditions on the same panels, same gain

In [ ]:
waterfall_overlay(RUNS, muscles=muscles, xlim=XLIM_WF, gains=gains, labels=LABELS);


## 5 · Every intensity, one figure each — traces + detected peaks + per-pulse peak-to-peak

Per muscle, for the intensity in the figure title:

- **top** — the two EMG traces overlaid, **gray / orange / blue / green = the conditions in `USE`, in order**.
  Red band = artifact of each pulse, green band = the response window.
  **▼ = the max, ▲ = the min** actually used for each pulse's peak-to-peak, in the trace's colour —
  this is the check that the right peaks are being detected.
- **middle** — peak-to-peak of *exactly those two traces*, pulse by pulse, **normalised to the
  first condition's pulse 1 = 100 %** (dotted line). The same reference is used for both
  conditions, so the first condition's pulse-1 bar is 100 % by construction and every other bar reads as a
  fraction of it. Dashed line = mean over the N pulses.
  **No error bars: one train per condition, nothing is averaged.** (`NORM_BARS = "none"` → mV.)
- **bottom** — the same numbers reduced to two bars: **pulse 1** vs the **mean of pulses 2–N**,
  same 100 % reference. The error bar on the mean is **± SD over pulses 2–N of that train**;
  pulse 1 is a single value so it has none. A later condition's pulse 1 below 100 % = a smaller first
  response than baseline; mean below pulse 1 = the train depresses.
- *below criterion* = no motor response; *ARTIFACT – rejected* = only artifact recovery in the
  window (see intro); *CLIPPED* = saturated channel.

The motor-threshold intensity (`AMP_MT`) is marked in its title. Figures are in increasing mA.
****

In [ ]:
NORM_BARS = "before_first"   # bars as % of the FIRST condition's pulse 1 (same reference for all)
                             # "none" -> mV instead

for a in AMPS:
    tag = f"{a} mA" + ("   ← motor threshold" if a == AMP_MT else "")
    compare_at_intensity(CSVS, None, amp=a, normalize=NORM_BARS, title=tag, labels=LABELS, **KW)


## 5b · Focus — one muscle, one intensity

Pick a muscle (by its label, e.g. `"Flex. carpi rad. (R)"`) and an intensity, and get the whole
analysis for that single case at full size:

1. the two traces (one colour per condition) with the response windows and the
   detected ▼/▲, per-pulse bars, pulse 1 vs mean 2–N — same layout as §5, one wide panel
2. the pulse overlay for each condition — the reliability check for exactly this case

`XLIM_FOCUS` sets the time range of the trace panel (default = the whole analysed train).

In [ ]:
MUSCLE     = "Ext. digitorum (R)"   # any label from the panel titles, or a channel name like "R_The_ED_FD_FCR D"
AMP_FOCUS  = AMP_SHOW                 # mA (must exist in both files)
XLIM_FOCUS = None                     # e.g. (-20, 120) for the first 4 pulses; None = whole analysed train

compare_at_intensity(CSVS, None, amp=AMP_FOCUS, normalize=NORM_BARS, muscles=MUSCLE,
                     xlim=XLIM_FOCUS, title=f"{MUSCLE} - {AMP_FOCUS} mA", labels=LABELS, **KW)
for lab, (meta, t, sig) in zip(LABELS, RUNS):
    plot_pulse_overlay(meta, t, sig, MUSCLE, amp=AMP_FOCUS, title=f"{AMP_FOCUS} mA - {lab}", **KW)


## 6 · Summary of all intensities — recruitment curves, nothing averaged

Per muscle, **x = stimulation intensity**. Gray, orange, blue, green = the conditions in `USE`, in order.

**Top row — recruitment.**
- **solid line, ● = pulse 1** peak-to-peak in mV. This is the classic recruitment curve.
- **dashed line, ▲ = mean of pulses 2–10** of the same train.
- The **vertical gap between ● and ▲ is the depression along the train**, in mV.
- **hollow ○ = below criterion**: no motor response at that intensity (drawn so you see where
  the threshold is; not used anywhere else).

**Bottom row — depression ratio.** Mean of pulses 2–10 as a % of pulse 1, only for trains with a
response. Dotted line = **100 % = no change**; below it the train depresses, above it facilitates.
Overlapping colours = the condition did not change how the train behaves.

How to read one muscle: Flex. digitorum (R) — ● rises from 30 mA (threshold) and gray/orange
overlap (lidocaine did not change recruitment); ▲ sits below ● (depression); the ratio climbs from
~10 % at threshold to ~80 % at 45 mA (less depression at higher intensity), the same in both.

In [ ]:
summary_curves(CSVS, None, labels=LABELS, **KW);


## 7 · Comparison with the original-polarity sessions

The earlier sessions (P04) were run with the device's default polarity. To see whether the
polarity changed anything, the pre-lidocaine cathodic and anodic files of this session are
plotted **together with one original-polarity reference file** — same colours-in-order logic.

`REF` picks the reference: the 16-07 Baseline (P04, **electrode 2 — same electrode number as
this session**, no lidocaine / vibration) or the 17-07 pre-lidocaine file (P04, electrode 1).
**Caveat: different participant** (P04 vs NTA), so absolute amplitudes and thresholds are not
expected to match; what is comparable is the *shape* — which muscles respond, the response
latency/waveform in the traces, and the depression along the train. Whichever of cathodic /
anodic looks like the reference is the polarity the original sessions used.

In [ ]:
REF_LABEL, REF_CSV, REF_MT = "original (P04 16-07, elec 2)", "tSCS_CHUV_data/16-07-2026/P04tscsHealthy/Burst_autosave_20260716_142311_232ms.csv", 40
# REF_LABEL, REF_CSV, REF_MT = "original (P04 17-07, elec 1)", "tSCS_CHUV_data/17-07-2026/P04tscsHealthy/Burst_autosave_20260717_150152_655ms.csv", 40

CMP_LABELS = [REF_LABEL] + [CONDITIONS[i][0] for i in (0, 1)]        # reference, pre cathodic, pre anodic
CMP_CSVS   = [REF_CSV]   + [D + CONDITIONS[i][1] for i in (0, 1)]
CMP_MT     = [REF_MT, AMP_MT, AMP_MT]                                # each file at its own motor threshold


### 7a · Waterfalls overlaid (same gain per muscle)

In [ ]:
RUNS_CMP = [load_run(f) for f in CMP_CSVS]
mus_cmp  = [c for c in RUNS_CMP[0][2] if c != "Trigger A" and all(c in r[2] for r in RUNS_CMP)]
g_cmp = waterfall(*RUNS_CMP[0], mus_cmp, xlim=XLIM_WF)
waterfall_overlay(RUNS_CMP, muscles=mus_cmp, xlim=XLIM_WF, gains=g_cmp, labels=CMP_LABELS);


### 7b · At motor threshold — traces, per-pulse p2p, pulse 1 vs rest (each file at its own MT)

In [ ]:
compare_at_intensity(CMP_CSVS, None, amp=CMP_MT, normalize="none", labels=CMP_LABELS,
                     title="each at its own motor threshold", **KW)


### 7c · Across intensities

In [ ]:
summary_curves(CMP_CSVS, None, labels=CMP_LABELS, **KW);
